# YOLOv9 Segmentation para Sistemas Fotovoltaicos en Imágenes Satelitales
Pipeline reproducible: configuración de entorno, descarga segura del dataset en formato polígonos (`yolov8`), entrenamiento y evaluación cuantitativa (mAP, mIoU, F1-Score, MAE, R²).

## 1. Verificación del Entorno y Montaje de Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!pip install -q ultralytics roboflow opencv-python scikit-learn matplotlib pandas

import ultralytics
ultralytics.checks()

## 2. Descarga del Dataset (Anotaciones Poligonales)
Para este caso usaremos un dataset elaborado por fuente propia de imagenes satelitales a un zoom de 19 en la plataforma SASPlanet con imagenes recortadas de 640x640

In [ ]:
import getpass
from roboflow import Roboflow

api_key = getpass.getpass('API Key: ')
rf = Roboflow(api_key=api_key)
project = rf.workspace('daves-workspace-cvhyt').project('satellite-pv')
version = project.version(2)
dataset = version.download('yolov8')
print(f'✓ Dataset descargado en: {dataset.location}')

## 3. Entrenamiento con YOLOv9c-seg

In [ ]:
import os
from ultralytics import YOLO

ruta_base = '/content/drive/MyDrive/Satelite_MID'
carpeta_resultados = os.path.join(ruta_base, 'Satelite_MID_YOLO_SEG')
os.makedirs(carpeta_resultados, exist_ok=True)

# Cargar modelo base de segmentación preentrenado
model = YOLO('yolov9c-seg.pt')

print(f'Iniciando entrenamiento de segmentación. Destino: {carpeta_resultados}')
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=1000,
    patience=50,
    imgsz=640,
    batch=16,
    project=carpeta_resultados,
    name='run_yolo_seg_poligonos',
    plots=True,
    save=True
)
print('✓ Entrenamiento completado.')

## 4. Evaluación Completa (Box/Mask mAP, F1-Score, mIoU y Regresión de Conteo)

In [ ]:
import os, cv2, numpy as np, pandas as pd
from ultralytics import YOLO
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

ruta_pesos = '/content/drive/MyDrive/Satelite_MID/Satelite_MID_YOLO_SEG/run_yolo_seg_poligonos/weights/best.pt'
carpeta_graficas = '/content/drive/MyDrive/Satelite_MID/Satelite_MID_YOLO_SEG/graficas_evaluacion'
os.makedirs(carpeta_graficas, exist_ok=True)

model = YOLO(ruta_pesos)

print('Validando split de prueba con métricas canónicas...')
metrics = model.val(data=f'{dataset.location}/data.yaml', split='val')

p_box, r_box = float(metrics.box.p[0]), float(metrics.box.r[0])
f1_box = 2 * (p_box * r_box) / (p_box + r_box + 1e-16)

p_seg, r_seg = float(metrics.seg.p[0]), float(metrics.seg.r[0])
f1_seg = 2 * (p_seg * r_seg) / (p_seg + r_seg + 1e-16)

# Rasterización y cálculo de IoU por imagen
val_images_dir = os.path.join(dataset.location, 'valid', 'images')
val_labels_dir = os.path.join(dataset.location, 'valid', 'labels')
image_files = sorted([f for f in os.listdir(val_images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

ious_list, y_true, y_pred = [], [], []

for img_file in image_files:
    img_path = os.path.join(val_images_dir, img_file)
    label_path = os.path.join(val_labels_dir, os.path.splitext(img_file)[0] + '.txt')
    
    img = cv2.imread(img_path)
    h, w, _ = img.shape
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    real_count = 0
    
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) > 5:
                    real_count += 1
                    coords = np.array([float(x) for x in parts[1:]], dtype=np.float32).reshape(-1, 2)
                    coords[:, 0] *= w
                    coords[:, 1] *= h
                    cv2.fillPoly(gt_mask, [coords.astype(np.int32)], 1)
    y_true.append(real_count)
    
    results = model.predict(img_path, conf=0.25, verbose=False)[0]
    pred_mask = np.zeros((h, w), dtype=np.uint8)
    pred_count = len(results.boxes) if results.boxes is not None else 0
    y_pred.append(pred_count)
    
    if results.masks is not None:
        for seg in results.masks.xy:
            if len(seg) > 0:
                cv2.fillPoly(pred_mask, [seg.astype(np.int32)], 1)
                
    intersection = np.logical_and(gt_mask, pred_mask).sum()
    union = np.logical_or(gt_mask, pred_mask).sum()
    iou = 1.0 if union == 0 and intersection == 0 else (intersection / union if union > 0 else 0.0)
    ious_list.append(iou)

ious_arr = np.array(ious_list)
miou = np.mean(ious_arr)
y_true, y_pred = np.array(y_true), np.array(y_pred)

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print('\n' + '='*55)
print('📊 RESULTADOS CONSOLIDADOS DE EVALUACIÓN')
print('='*55)
print(f'• Box Precision:              {p_box:.4f}')
print(f'• Box Recall:                 {r_box:.4f}')
print(f'• Box F1-Score:               {f1_box:.4f}')
print(f'• Box mAP@50:                 {metrics.box.map50:.4f}')
print(f'• Box mAP@50-95:              {metrics.box.map:.4f}')
print('-' * 55)
print(f'• Mask Precision:             {p_seg:.4f}')
print(f'• Mask Recall:                {r_seg:.4f}')
print(f'• Mask F1-Score:              {f1_seg:.4f}')
print(f'• Mask mAP@50:                {metrics.seg.map50:.4f}')
print(f'• Mask mAP@50-95:             {metrics.seg.map:.4f}')
print('-' * 55)
print(f'• mIoU (Mean IoU Máscaras):   {miou:.4f} ({miou*100:.2f}%)')
print(f'• MAE (Conteo Paneles):       {mae:.4f} paneles/imagen')
print(f'• RMSE:                       {rmse:.4f}')
print(f'• R² (Determinación):         {r2:.4f}')
print('='*55 + '\n')

## 5. Exportación de Figuras Individuales en Alta Resolución

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 11, 'figure.autolayout': True})

# Gráfica 1: Dispersión Real vs Predicho
plt.figure(figsize=(7, 6))
plt.scatter(y_true, y_pred, color='#1f77b4', alpha=0.7, edgecolors='k', s=55, label='Predicciones YOLO-Seg')
limite_max = max(y_true.max(), y_pred.max()) + 2
plt.plot([0, limite_max], [0, limite_max], '--r', linewidth=2, label='Ajuste Ideal ($y=x$)')
plt.title(f'Conteo de Paneles: Real vs Predicho\n($R^2 = {r2:.3f}$, $MAE = {mae:.2f}$)', pad=12)
plt.xlabel('Paneles Reales (Ground Truth)')
plt.ylabel('Paneles Predichos')
plt.xlim(0, limite_max)
plt.ylim(0, limite_max)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper left')
plt.savefig(os.path.join(carpeta_graficas, '1_dispersion_real_vs_pred.png'), dpi=300)
plt.show()

# Gráfica 2: Distribución de IoU
plt.figure(figsize=(7, 5))
plt.hist(ious_arr, bins=15, color='#2ca02c', edgecolor='black', alpha=0.8)
plt.axvline(miou, color='red', linestyle='--', linewidth=2, label=f'mIoU Promedio = {miou:.3f}')
plt.title('Distribución de IoU de Segmentación por Imagen', pad=12)
plt.xlabel('Intersección sobre Unión (IoU)')
plt.ylabel('Cantidad de Muestras')
plt.xlim(0, 1.0)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.savefig(os.path.join(carpeta_graficas, '2_distribucion_iou.png'), dpi=300)
plt.show()

# Gráfica 3: Distribución de Residuales
errores = y_pred - y_true
plt.figure(figsize=(7, 5))
plt.hist(errores, bins=15, color='#3b528b', edgecolor='black', alpha=0.8)
plt.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Error Cero')
plt.title('Distribución de Residuales ($y_{pred} - y_{real}$)', pad=12)
plt.xlabel('Error Residual (Paneles)')
plt.ylabel('Frecuencia (Imágenes)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.savefig(os.path.join(carpeta_graficas, '3_distribucion_residuales.png'), dpi=300)
plt.show()

print(f'✓ Gráficas guardadas en: {carpeta_graficas}')